In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import pltkit
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shapereader
import matplotlib.ticker as mticker

### ERF

In [ ]:
# Map of the ERF as in Honda et al., (2014)
wdir = "X:\\user\\liprandicn\\mt-comparison\\Honda2014\\"

df_interp = pd.read_csv(wdir + "data\\risk_function\\RiskFunction_Honda.csv")

fig, ax = plt.subplots(figsize=(8,6))
ax.plot(df_interp["daily_temperature"], df_interp["relative_risk_mean"], color="k")
ax.fill_between(
    df_interp["daily_temperature"], 
    df_interp["relative_risk_lower"], 
    df_interp["relative_risk_upper"], 
    alpha=0.2, color="k")
pltkit.StylizeAxes(
    ax=ax,
    spines={"top":False, "right":False},
    xlabel="Daily temperature (°C) - OT",
    ylabel="Relative Risk",
    title="Risk function of excess mortality due to heat",
    title_kwargs={"fontsize": 12, "y":1.05},
)

### Optimal temperatures

In [ ]:
# Map of optimal temperatures (84th percentile of daily maximum temperatures) from ERA5 reanalysis for 1980-2010 period.

wdir = "X:\\user\\liprandicn\\mt-comparison\\honda2014/data/"
ot = xr.open_dataset(wdir+"optimal_temperatures/era5_t2m_max_1980-2010_p84.nc")

fig = plt.figure(figsize=(10,5), dpi=300)
ax = fig.add_subplot(111, projection=ccrs.Robinson(central_longitude=0), frameon=True)
ax.coastlines(resolution='10m', lw = 0.1)
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0, color='white', alpha=0.5)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 6}
gl.ylabel_style = {'size': 6}
im = ot.t2m_p84.plot(ax=ax, 
                transform=ccrs.PlateCarree(), 
                cmap='RdBu_r', 
                add_colorbar=False,
                levels=21, 
                vmin=-40, 
                vmax=40)
ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=2)

# Manually set colorbar limits
cbar = fig.colorbar(im, ax=ax, orientation='vertical', shrink=0.6, pad=0.05, aspect=20)
# cbar.set_label(r'Daily maximum temperature', fontsize=8)
cbar.ax.set_ylim(-12, 36) 
cbar.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%d°C'))

# Remove Antarctica 
land_shp = shapereader.natural_earth(resolution='110m', category='physical', name='land')
land_geoms = shapereader.Reader(land_shp).geometries()
for land in land_geoms:
    if land.bounds[1] < -60:
        ax.add_geometries([land], ccrs.PlateCarree(), 
                          facecolor='white', 
                          edgecolor='none', 
                          zorder=2)
        
plt.title('Optimal daily maximum temperatures (OT)', fontsize=10)

plt.show()

### Replication Romanello

In [ ]:
wdir = "X:\\user\\liprandicn\\mt-comparison\\honda2014/output/"
filename = "ReplicationRomanello/mortality_ReplicationRomanello_SSP2_ERA5_2000-2023"

rt = "IMAGE"
temp_type = "heat"
age_group = "oldest"
region = "World"
cause = "All causes"
var = "mortality"

rom_mean, rom_upper, rom_lower = pltkit.LoadMortality(wdir, filename, rt, region, temp_type, cause, age_group, var)

fig = plt.figure(figsize=(12,5), dpi=300)
plt.plot(rom_mean.year, rom_mean.values, label="Romanello et al., 2024", linewidth=2, c="k")
plt.fill_between(rom_mean.year, rom_lower.values, rom_upper.values, alpha=0.2, color="k")

formatter = mticker.ScalarFormatter(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((5, 5))
plt.gca().yaxis.set_major_formatter(formatter)
plt.grid(False)
plt.legend()
plt.ylim(0, 5e5)
plt.title("Heat related mortality")
plt.ylabel("Excess mortality over 65")
plt.show()